In [1]:
import numpy as np
import gym
from gym import spaces
import random
from gym.utils import seeding



class conn(gym.Env):
    
  """
  Custom Environment that follows gym interface.
  This is a simple env where the agent must learn to go always left.
  """
  # Because of google colab, we cannot implement the GUI ('human' render mode)
  metadata = {'render.modes': ['console']}
  # Define constants for clearer code


  def __init__(self,connectors_dicc, list_signals, connector, len_test_conn, max_categories, state_space=10000):
      super(conn, self).__init__()
      # numero de pins connector -1 due to PartNumber
      self.grid_size=len_test_conn
      self.connectors_dicc=connectors_dicc
      self.signals=list_signals # different signals to plug into a PIN Categories
      # Example when using discrete actions,
      self.state_space=state_space
      self.max_categories=max_categories
      # space number of pins of this connector
      self.action_space = spaces.Discrete(self.state_space)
      # observation space low/High bound number of diferent signals this connector
      # size, vector length number of pins
      self.observation_space = spaces.Box(low=0, high=self.max_categories,
                                        shape=(self.grid_size,), dtype=np.float32)
      # estate list of actions
      self.state = list(np.zeros(self.grid_size, dtype='int'))
      self.agent_pos=0
      self.terminal= False
      self.wirecolors=[]
      self.signal_groups=[]
      self.lr=.5
      self.this_connector=connector
      self.reward = 0
      self.already_pinned=[]
      self.already_pinned_thickness=[]
      self._seed()
      self.signals_cats=[]
      self.already_pinned_in_out=[]
      self.multicore_pinned=[]
      self.signals_category_dicc=dicci=eval(self.connectors_dicc[self.this_connector]['1']['dicc_signals'])
      for key in self.connectors_dicc[self.this_connector].keys():
            # list with real categories of this conector 
            if key !="PartNumber":
                self.signals_cats.append(self.connectors_dicc[self.this_connector][key]['Cat_Signal'])
      return        
        
  def _seed(self, seed=None):
      self.np_random, seed = seeding.np_random(seed)
      return [seed]

  def reset(self):
      """
      Important: the observation must be a numpy array
      :return: (np.array)
      :return:
      """
      self.state = np.zeros(self.grid_size, dtype='int')

      self.agent_pos=0
      self.reward=0
      self.wirecolors=[]
      self.signal_groups=[]
      self.already_pinned=[]
      self.already_pinned_thickness=[]
      self.already_pinned_in_out=[]
      self.multicore_pinned=[]

      return self.state

  def render(self, mode='console'):
      if mode != 'console':
          raise NotImplementedError()
      # agent is represented as a cross, rest as a dot


  
  def get_reward_cat(self, reward, agent_pos, signal_group, neighbors):
        # save the previous rewards
        old_reward=reward
        for pin in neighbors:
            if pin in self.already_pinned:
                # get signal group of this neigbourg
                # 'BATT_'1, 'CAN_' 2, 'CANFD_' 3, 'GND_' 4, 'GNDR_GND_' 5,'HV_' 6, else 7
                signal_group_prev=self.connectors_dicc[self.this_connector][str(pin)]['signal_group']
                                
                #Don't put power and ground next to each other   # 1--> battery, 6--> High Power 4,5 Ground
                if (signal_group_prev==1 and signal_group==5) or (signal_group_prev==1 and signal_group==4):
                    reward = reward -4
                elif (signal_group_prev==5 and signal_group==1) or (signal_group_prev==4 and signal_group==1):
                    reward = reward -4
                if (signal_group_prev==6 and signal_group==5) or (signal_group_prev==6 and signal_group==4):
                    reward = reward -8
                elif (signal_group_prev==5 and signal_group==6) or (signal_group_prev==4 and signal_group==6):
                    reward = reward -8
                    
                #Don't put power and information next to each other   # 1--> battery, 6--> High Power 2,3 Ground    
                if (signal_group_prev==1 and signal_group==3) or (signal_group_prev==1 and signal_group==2):
                    reward = reward -4
                elif (signal_group_prev==3 and signal_group==1) or (signal_group_prev==2 and signal_group==1):
                    reward = reward -4
                if (signal_group_prev==6 and signal_group==3) or (signal_group_prev==6 and signal_group==2):
                    reward = reward -8
                elif (signal_group_prev==3 and signal_group==6) or (signal_group_prev==2 and signal_group==6):
                    reward = reward -8    
                    

        if old_reward==reward:
            reward = reward + 8
        return reward
  def get_reward_colorwire(self, reward, agent_pos, colorwire, neighbors, thickness):
        # save the previous rewards
        old_reward=reward
        for pin in neighbors:
            if pin in self.already_pinned:
                if colorwire !="na" and colorwire in self.wirecolors and thickness in self.already_pinned_thickness :
                    reward = reward - 4
                elif colorwire !="na" and colorwire in self.wirecolors and not( thickness in self.already_pinned_thickness):
                    reward = reward + 2
                elif colorwire !="na":
                    reward= reward + 1
                
        if old_reward==reward:
            reward = reward + 8
        return reward
  
  def get_reward_multicore(self, reward, agent_pos, multicore_label, neighbors, in_ext):
        # save the previous rewards
        old_reward=reward
        for pin in neighbors:
            if pin in self.already_pinned:
                # multicore with pair in neigbhors and pining to outside
                if multicore_label !="na" and multicore_label in self.multicore_pinned and in_ext =="False":
                    reward = reward + 4
                # multicore with pair in neigbhors and NOT pining to outside
                elif multicore_label !="na" and multicore_label in self.multicore_pinned and in_ext =="True":
                    reward = reward -4
                # multicore with NOT pair in neigbhors and pining to outside
                elif multicore_label !="na" and not(multicore_label in self.multicore_pinned) and in_ext =="False" :
                    reward = reward - 2
                elif multicore_label !="na" and not(multicore_label in self.multicore_pinned) and in_ext =="True" :
                    reward = reward - 4
                elif multicore_label !="na":
                    reward= reward + 1
        # if not multicore on the neighbors lets assume that is a new multicore        
        if old_reward==reward and multicore_label !="na":
            reward = reward + 4
        return reward
  def get_reward_internal_external(self, reward, agent_pos, in_ext, thickness):
        # Thick wires outside and thin wires internal if exists internal cavities
        # we conside a thick wire > 1 
        
        if in_ext=="False" and thickness <= 1:
            reward = reward -2
        
        elif in_ext=="True" and thickness <= 1:
            reward = reward + 2
        elif in_ext=="False" and thickness > 1:
            reward = reward + 2
        elif in_ext=="True" and thickness > 1:
            reward = reward -1
        return reward
      
    
  def calculate_reward(self,combination):
    reward=0

    signal=combination[self.agent_pos] #signal category (action = pin this wire with this signal)
    # check if this signal was pinned in this Pin
    
    if self.connectors_dicc[self.this_connector][str(self.agent_pos+1)]['Cat_Signal']==signal:
        reward = reward + 10
    else:
        reward = reward - 10
    
    # WireColor
    colorwire=self.connectors_dicc[self.this_connector][str(self.agent_pos+1)]['WireColor']
    # thickness
    thickness=self.connectors_dicc[self.this_connector][str(self.agent_pos+1)]['WireCSA']
    # signal Group
    signal_group=self.connectors_dicc[self.this_connector][str(self.agent_pos+1)]['signal_group']
    # neighbors
    neighbors=self.connectors_dicc[self.this_connector][str(self.agent_pos+1)]['neighbors']
    # internal/external PIN
    in_ext = self.connectors_dicc[self.this_connector][str(self.agent_pos+1)]['internal_pin']
    #Mulicore Features
    multicore_label = self.connectors_dicc[self.this_connector][str(self.agent_pos+1)]['Multicore']
    
    if self.agent_pos+1 >1:
        # get position signal reward
        reward = self.get_reward_cat(reward, self.agent_pos+1, signal_group, neighbors)
        # get colorwire reward
        reward = self.get_reward_colorwire(reward, self.agent_pos+1, colorwire, neighbors, thickness)
        # get internal/external rewars
        reward = self.get_reward_internal_external(reward, self.agent_pos+1, in_ext, thickness)
        # get Mukticore reward
        reward = self.get_reward_multicore(reward, self.agent_pos+1, multicore_label, neighbors, in_ext)
        
    return reward

  
  def get_real(self, action):
    real=0
    if str(action) in self.signals_cats:
        real=self.signals_cats.index(action)+1
    else:
        real=self.agent_pos+1
    return real

  def step(self, action):
      
    #Update Action    
    self.state[self.agent_pos]=action
    real=self.get_real(action) #  real values
    #calculate Reward
    self.reward = self.reward+self.calculate_reward(self.state)
    print(self.reward)
    
    self.already_pinned.append(self.agent_pos+1)# list control which pin are already in use
    #self.terminal = bool(self.agent_pos >= 1000) or self.reward > self.grid_size*7
    self.wirecolors.append(self.connectors_dicc[self.this_connector][str(real)]['WireColor'])
    self.signal_groups.append(self.connectors_dicc[self.this_connector][str(real)]['signal_group'])
    self.already_pinned_thickness.append(connectors_dicc[self.this_connector][str(real)]['WireCSA'])
    self.already_pinned_in_out.append(connectors_dicc[self.this_connector][str(real)]['internal_pin'])
    self.multicore_pinned.append(connectors_dicc[self.this_connector][str(real)]['Multicore'])
    # Account for the boundaries of the grid
    self.agent_pos =  self.agent_pos + 1
    self.terminal = bool(self.agent_pos > self.grid_size -1)

    # Optionally we can pass additional info, we are not using that for now
    info = {}

    return self.state, self.reward, self.terminal, info


In [2]:
from agent_pinner.pre_processing import *
import warnings

import pandas as pd
from pandas.core.common import SettingWithCopyWarning

warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)

In [106]:
data1 = pd.read_csv("agent_pinner/data/symbol_pins.csv")
symbols=get_neighbors_partnumbers(data1)

data = pd.read_csv("agent_pinner/data/inline_pins.csv")
data, d, _ = preprocess_data(data, symbols)

connectors_dicc=create_connector_dicc(data)
# observation space is a vector length number of pins. each element is an int to choose among the list of
# signal categories


In [4]:
#(13) TO_121_TNGA_F/219_13P  #(3) TO_111_LH_F/143_3P    (13)   TO_161_RH_F/169_13P  (13) TO_169_F/161_RH_13P

In [107]:
connector="TO_161_RH_F/169_13P"

In [108]:
len_test_conn=len(connectors_dicc[connector].keys()) - 1

In [109]:
data1=data[data['Connector Name']==connector]

list_signals=list(data1['Cat_Signal Name'].unique())
# state space is number of pins of this connector * list signals available to pin in the connector


list_signals

[13, 19, 17, 21, 27, 12, 18, 16, 22, 20]

In [110]:
state_space =len(eval(connectors_dicc[connector]['1']['dicc_signals']).keys())

In [111]:
state_space,len_test_conn

(28, 13)

In [112]:
#connectors_dicc[connector]

In [113]:
signals_cats=[]
for key in connectors_dicc[connector].keys():
    if key !="PartNumber":
        signals_cats.append(connectors_dicc[connector][key]['Cat_Signal'])
        
signals_cats

[13, 19, 17, 21, 27, 27, 12, 18, 16, 22, 20, 27, 27]

In [114]:
from stable_baselines3.common.env_checker import check_env

In [115]:
max_categories=(len(eval(connectors_dicc[connector]['1']['dicc_signals']).keys()))
max_categories

28

In [116]:
env = conn(connectors_dicc,list_signals ,connector , len_test_conn, max_categories,state_space)

In [117]:
check_env(env, warn=True)

-10
-2
6
14
18
22
30
38
46
54
62


In [118]:
obs = env.reset()
env.render()

In [119]:
print(env.observation_space)
print(env.action_space)
print(env.action_space.sample())

Box(0.0, 28.0, (13,), float32)
Discrete(28)
6


In [120]:
n_steps = 1300
GO_LEFT=random.randint(0, max_categories )
for step in range(n_steps):
    #print("Step {}".format(step + 1))
    GO_LEFT=random.randint(0, max_categories )
    obs, reward, done, info = env.step(GO_LEFT)
    #print('obs=', obs, 'reward=', reward, 'done=', done)
    env.render()
    if done:
        print("Goal reached!", "reward=", reward)
        obs = env.reset()
        

-10
-2
6
14
18
22
30
38
46
54
62
66
74
Goal reached! reward= 74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
Goal reached! reward= 74
-10
-2
6
14
18
22
30
58
66
74
82
106
114
Goal reached! reward= 114
-10
-2
6
14
18
22
30
38
46
54
62
66
74
Goal reached! reward= 74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
Goal reached! reward= 74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
Goal reached! reward= 74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
Goal reached! reward= 74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
Goal reached! reward= 74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
Goal reached! reward= 74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
Goal reached! reward= 74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
Goal reached! reward= 74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
Goal reached! reward= 74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
Goal reached! reward= 74
-10
-2
6
14
18
22
30
38
46
54
62
86
94
Goal reached! reward= 94
-10
-2
6
14
18
22
30
38
46
54
62
66
74
Goal reached! reward= 74
-10
18
26
54
58
62
70
78
86
94
102
10

In [121]:
from stable_baselines3.common.vec_env import DummyVecEnv

In [122]:
# 13 TO_161_RH_F/169_13P  TO_121_TNGA_F/219_13P

In [123]:
env_train = DummyVecEnv([lambda: conn(connectors_dicc,list_signals ,connector ,len_test_conn, max_categories,state_space)])

In [124]:
from stable_baselines3 import DQN, TD3

In [125]:
model = DQN('MlpPolicy', env_train, verbose=2)

Using cpu device


In [126]:
#model = TD3('MlpPolicy', env_train, verbose=2)

In [127]:
import time
time.time()

1625639405.356652

In [128]:
time1=time.time()
model.learn(20000)
time2=time.time()

print(time2-time1)

-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
58
66
74
82
86
94
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
----------------------------------
| rollout/            |          |
|    exploration rate | 0.975    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 1180     |
|    time_elapsed     | 0        |
|    total timesteps  | 52       |
----------------------------------
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
58
66
74
82
86
94
----------------------------------
| rollout/            |          |
|    exploration rate | 0.951    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 1155     |
|    time_elapsed     | 0        |
|    total timesteps  | 104      |
----------------------------------
-10
-2
6
14
18
42
50
58
86
94
102
106
114
-10
-2
6
14
18
2

-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
50
58
66
74
82
86
94
-10
-2
6
14
18
42
50
58
86
94
102
106
114
-10
-2
6
14
18
22
30
38
46
54
62
66
74
----------------------------------
| rollout/            |          |
|    exploration rate | 0.531    |
| time/               |          |
|    episodes         | 76       |
|    fps              | 1118     |
|    time_elapsed     | 0        |
|    total timesteps  | 988      |
----------------------------------
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
38
42
50
58
66
74
82
86
94
-10
-2
6
14
18
22
30
38
46
54
62
66
74
----------------------------------
| rollout/            |          |
|    exploration rate | 0.506    |
| time/               |          |
|    episodes         | 80       |
|    fps              | 1116     |
|    time_elapsed     | 0        |
|    total timesteps  | 1040     |
----------------------------------
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
38
4

-10
-2
6
14
18
22
30
38
46
54
62
66
94
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
58
86
94
102
106
114
-10
-2
6
14
18
22
30
38
46
54
62
66
74
----------------------------------
| rollout/            |          |
|    exploration rate | 0.0861   |
| time/               |          |
|    episodes         | 148      |
|    fps              | 1170     |
|    time_elapsed     | 1        |
|    total timesteps  | 1924     |
----------------------------------
-10
-2
6
14
38
42
50
58
66
74
82
86
94
-10
-2
6
14
18
42
50
58
66
74
82
86
94
-10
-2
6
34
38
42
50
58
66
74
82
86
94
-10
-2
6
14
18
22
30
38
46
54
62
66
74
----------------------------------
| rollout/            |          |
|    exploration rate | 0.0614   |
| time/               |          |
|    episodes         | 152      |
|    fps              | 1170     |
|    time_elapsed     | 1        |
|    total timesteps  | 1976     |
----------------------------------
-10
-2
6
14
38
42
50
58
66
74
82
86
94
-10
-2
6
14
18
4

-10
-2
6
14
18
22
30
38
46
54
62
86
94
-10
-2
6
14
18
22
30
38
46
54
62
66
94
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
----------------------------------
| rollout/            |          |
|    exploration rate | 0.05     |
| time/               |          |
|    episodes         | 220      |
|    fps              | 1195     |
|    time_elapsed     | 2        |
|    total timesteps  | 2860     |
----------------------------------
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
26
34
38
42
50
58
66
74
82
86
94
-10
-2
6
14
18
22
30
38
46
54
62
66
74
----------------------------------
| rollout/            |          |
|    exploration rate | 0.05     |
| time/               |          |
|    episodes         | 224      |
|    fps              | 1194     |
|    time_elapsed     | 2        |
|    total timesteps  | 2912     |
----------------------------------
-10
-2
6
14
18
22
50
78
86
94
102
106
114
-10
-2
6
14
18


10
18
26
34
38
42
50
58
66
74
82
86
94
-10
-2
6
14
18
22
30
38
46
54
82
86
114
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
----------------------------------
| rollout/            |          |
|    exploration rate | 0.05     |
| time/               |          |
|    episodes         | 292      |
|    fps              | 1224     |
|    time_elapsed     | 3        |
|    total timesteps  | 3796     |
----------------------------------
-10
-2
6
14
18
42
50
58
66
74
82
86
94
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
----------------------------------
| rollout/            |          |
|    exploration rate | 0.05     |
| time/               |          |
|    episodes         | 296      |
|    fps              | 1225     |
|    time_elapsed     | 3        |
|    total timesteps  | 3848     |
----------------------------------
-10
-2
6
34
38
42
50
58
66
74
82
86
94
-10
-2
6
14
18
22


...

-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
38
42
50
58
66
74
82
86
94
----------------------------------
| rollout/            |          |
|    exploration rate | 0.05     |
| time/               |          |
|    episodes         | 1228     |
|    fps              | 1285     |
|    time_elapsed     | 12       |
|    total timesteps  | 15964    |
----------------------------------
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
10
18
26
34
38
42
50
58
66
74
82
86
94
-10
-2
6
14
18
22
30
38
46
54
62
66
74
----------------------------------
| rollout/            |          |
|    exploration rate | 0.05     |
| time/               |          |
|    episodes         | 1232     |
|    fps              | 1285     |
|    time_elapsed     | 12       |
|    total timesteps  | 16016    |
----------------------------------
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
3

-10
-2
6
14
18
22
30
38
46
54
62
66
74
10
38
46
54
58
62
70
78
86
94
102
106
114
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
----------------------------------
| rollout/            |          |
|    exploration rate | 0.05     |
| time/               |          |
|    episodes         | 1300     |
|    fps              | 1289     |
|    time_elapsed     | 13       |
|    total timesteps  | 16900    |
----------------------------------
-10
-2
6
14
18
22
30
58
66
74
102
106
114
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
----------------------------------
| rollout/            |          |
|    exploration rate | 0.05     |
| time/               |          |
|    episodes         | 1304     |
|    fps              | 1290     |
|    time_elapsed     | 13       |
|    total timesteps  | 16952    |
----------------------------------
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
1

-10
-2
6
14
18
22
50
58
66
74
82
86
94
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
42
50
58
66
74
82
86
94
----------------------------------
| rollout/            |          |
|    exploration rate | 0.05     |
| time/               |          |
|    episodes         | 1372     |
|    fps              | 1289     |
|    time_elapsed     | 13       |
|    total timesteps  | 17836    |
----------------------------------
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
18
26
34
38
42
50
58
66
74
82
86
114
----------------------------------
| rollout/            |          |
|    exploration rate | 0.05     |
| time/               |          |
|    episodes         | 1376     |
|    fps              | 1288     |
|    time_elapsed     | 13       |
|    total timesteps  | 17888    |
----------------------------------
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
38
42

-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
18
26
34
38
42
50
58
66
74
82
86
94
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
26
34
38
42
50
58
66
74
82
86
94
----------------------------------
| rollout/            |          |
|    exploration rate | 0.05     |
| time/               |          |
|    episodes         | 1444     |
|    fps              | 1289     |
|    time_elapsed     | 14       |
|    total timesteps  | 18772    |
----------------------------------
-10
-2
6
14
18
22
30
58
66
74
82
86
114
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
74
82
86
94
-10
-2
6
14
18
22
30
38
46
54
62
66
74
----------------------------------
| rollout/            |          |
|    exploration rate | 0.05     |
| time/               |          |
|    episodes         | 1448     |
|    fps              | 1288     |
|    time_elapsed     | 14       |
|    total timesteps  | 18824    |
----------------------------------
10
18
26
34
38
42
50
78
86
94
102
106
114
-10
-2
6
14
1

-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
34
38
42
50
58
66
74
82
86
94
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
----------------------------------
| rollout/            |          |
|    exploration rate | 0.05     |
| time/               |          |
|    episodes         | 1516     |
|    fps              | 1287     |
|    time_elapsed     | 15       |
|    total timesteps  | 19708    |
----------------------------------
-10
-2
6
34
38
42
50
58
66
74
82
86
94
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
18
22
30
38
46
54
62
66
74
-10
-2
6
14
38
42
50
58
66
74
82
86
94
----------------------------------
| rollout/            |          |
|    exploration rate | 0.05     |
| time/               |          |
|    episodes         | 1520     |
|    fps              | 1287     |
|    time_elapsed     | 15       |
|    total timesteps  | 19760    |
----------------------------------
-10
18
26
34
38
42
50
58
66
74
82
86
94
-10
-2
6
14
18
22


In [129]:
obser=[6,7,8,4,3,5,0,1,2,8,8,8,8,]#'TO_121_TNGA_F/219_13P'
# obser=[0,2,1] #TO_111_LH_F/143_3P
# #obser=[14,15,27,6,5,7,2,3,4,27,27,27,27] #TO_121_TNGA_F/219_13P   TO_161_RH_F/169_13P 

# obser=[13, 19, 17, 21, 27, 27, 12, 18, 16, 22, 20, 27, 27]#'TO_169_F/161_RH_13P'
# obser=[14, 15, 27, 6, 5, 7, 2, 3, 4, 27, 27, 27, 27]#'TO_121_TNGA_F/219_13P'
# obser= signals_cats #TO_169_F/161_RH_13P
obser =[13, 19, 17, 21, 27, 27, 12, 18, 16, 22, 20, 27, 27]
obser=[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
obser=[27, 27, 27, 27, 14, 15, 27, 6, 5, 7, 2, 3, 4]
obser=[26, 26, 26, 26, 14, 15, 27, 6, 5, 7, 2, 3, 4]

In [130]:
signals_cats

[13, 19, 17, 21, 27, 27, 12, 18, 16, 22, 20, 27, 27]

In [131]:
obs = env.reset()
env.render()

In [132]:
model.env.reset()
new_observ=list(np.zeros(len(obser), dtype='int'))
reward=0
for x in range(0,len(obser)):
    new_observ[x]=obser[x]
    #print(new_observ)
    pred=model.predict(np.array(new_observ), deterministic= False)
    print(pred)
    reward=reward+pred[0]
print(reward)

(18, None)
(8, None)
(9, None)
(9, None)
(9, None)
(9, None)
(9, None)
(9, None)
(9, None)
(array(21), None)
(array(19), None)
(9, None)
(9, None)
147


In [56]:
connector

'TO_161_RH_F/169_13P'

In [ ]:
'TO_169_F/161_RH_13P' =234 , 'TO_121_TNGA_F/219_13P' = 342 'TO_169_F/161_RH_13P' = 113, "TO_111_LH_F/143_3P" =0

### EVALUATION

In [133]:
import pickle
with open("agent_pinner/conf/connectors_dicc.pkl", 'rb') as file:
    dicc = pickle.load(file)

In [134]:
dicc

{'TO_111_LH_F/113': {'1': {'PreferredSignal': 'na',
   'Cat_PreferredSignal': 18,
   'Signal': 'EMB_PVM_ECU_064',
   'Cat_Signal': 3,
   'dicc_signals': "{0: 'CLEARANCE_SNR_ECU_1444', 1: 'CLEARANCE_SNR_ECU_1456', 2: 'EMB_FRONT_CAMERA-PVM_063', 3: 'EMB_PVM_ECU_064', 4: 'EMB_PVM_ECU_065', 5: 'EMB_PVM_ECU_066', 6: 'EMB_PVM_ECU_067', 7: 'EQ_PDB_TAIL', 8: 'GND_ACC_LP_LH_E', 9: 'TSS3_RADAR_0680', 10: 'TSS3_RADAR_0685', 11: 'TW_CERTIFICATION_ECU_278', 12: 'TW_CERTIFICATION_ECU_279', 13: 'TW_CLEARANCE_SNR_ECU_075', 14: 'TW_CLEARANCE_SNR_ECU_076', 15: 'TW_CLEARANCE_SNR_ECU_077', 16: 'TW_CLEARANCE_SNR_ECU_078'}",
   'check_signal': 0,
   'Num_Pins': 17,
   'Num_Pin_unique': 17,
   'more_signal_in_pin': False,
   'WireColor': 'W',
   'WireCSA': 1.0,
   'Multicore': 'MC137112',
   'signal_group': 7,
   'internal_pin': nan,
   'neighbors_length': nan,
   'mean_neighbors': nan,
   'centroide_distance': nan,
   'number_internal_pin': nan,
   'max_distance': nan,
   'min_distance': nan,
   'Cat_Wire_W